# Notebook 2: GEDI UOI Signal Generation (GEE)

This notebook computes the raw Understory Openness Index (UOI) from GEDI L2B data.
It follows a strict modular structure:
1. Setup & Configuration
2. Methodological Logic (Functions)
3. Unit Tests
4. Visual Integration Test
5. Execution (Asset Export)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee
import geemap

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")

# Output destination
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study Regions
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
STUDY_REGION = ee.FeatureCollection([
    ee.Feature(CONGO_BBOX, {'basin': 'Congo'}),
    ee.Feature(AMAZON_BBOX, {'basin': 'Amazon'})
])

# Scales (meters)
SCALES = list(range(5000, 105000, 5000))

# Datasets
GEDI_L2B = 'LARSE/GEDI/GEDI02_B_002_MONTHLY'
FOREST_MASK = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS = 10
FOREST_COVER_THRESHOLD = 0.95
MODIS_SCALE = 500  # Harmonizing resolution with NB1 FRIP mask
SRTM = 'USGS/SRTMGL1_003'

# Date range
START_DATE = '2020-01-01'
END_DATE = '2023-12-31'

print("✓ Configuration loaded.")

In [ ]:
# =============================================================================
# BLOCK 2: METHODOLOGICAL LOGIC (FUNCTIONS)
# =============================================================================

def load_base_masks():
    """Loads JRC TMF and SRTM layers. Builds a dual-layer pristine forest mask.
    
    Layer 1 (Native 30m): Binary intact forest AND flat topography mask.
    Layer 2 (500m MODIS-harmonized): Only 500m pixels where >=95% of underlying
    30m pixels are intact forest. This ensures GEDI footprints are only retained
    from deep core forest, not edge-effect zones, and is identical to the mask
    applied to MODIS NPP in Notebook 1 (FRIP).
    """
    # Load 30m JRC TMF
    tmf_col = ee.ImageCollection(FOREST_MASK)
    tmf = tmf_col.mosaic().setDefaultProjection(tmf_col.first().projection())
    forest_mask_30m = tmf.eq(FOREST_CLASS)
    
    # Topography (Elevation < 1000m, Slope < 10 deg)
    srtm = ee.Image(SRTM)
    elev = srtm.select('elevation')
    slope = ee.Terrain.slope(srtm)
    topo_mask = elev.lt(1000).And(slope.lt(10))
    
    # Native 30m combined mask (forest + topo)
    native_mask = forest_mask_30m.updateMask(topo_mask)
    
    # Aggregate 30m forest mask to 500m MODIS resolution (~277 pixels, under 65535 limit)
    modis_proj = ee.Projection('EPSG:4326').atScale(MODIS_SCALE)
    forest_fraction_500m = forest_mask_30m.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).reproject(crs=modis_proj)
    
    # Pristine forest mask at 500m: >=95% intact forest cover
    pristine_mask_500m = forest_fraction_500m.gte(FOREST_COVER_THRESHOLD)
    
    # Combined dual-layer mask: native 30m precision AND 500m landscape purity
    combined_mask = native_mask.updateMask(pristine_mask_500m)
    
    return combined_mask

def compute_native_uoi(combined_mask):
    """Calculates native resolution UOI and observation counts from GEDI."""
    # Load GEDI and filter by date
    gedi = ee.ImageCollection(GEDI_L2B).filterDate(START_DATE, END_DATE)
    
    def calc_uoi(img):
        pai = img.select('pai')
        pavd_z0 = img.select('pavd_z0')
        
        # UOI = 1 - (pavd_z0 / pai)
        uoi = ee.Image(1).subtract(pavd_z0.divide(pai))
        # Clip to [0, 1] range to handle anomalies
        uoi = uoi.clamp(0, 1).rename('UOI')
        return uoi.updateMask(combined_mask)
    
    # Map UOI calculation over collection
    uoi_col = gedi.map(calc_uoi)
    
    # Calculate mean UOI and count N (number of valid footprint observations)
    mean_uoi = uoi_col.mean().rename('UOI_mean')
    count_n = uoi_col.count().rename('N')
    
    # Stack the bands
    native_proj = gedi.first().projection()
    native_stack = ee.Image.cat([mean_uoi, count_n]).setDefaultProjection(native_proj)
    
    return native_stack, native_proj

def build_gedi_asset(scale):
    """Aggregates UOI to the target scale. Does NOT call reproject() — the Export
    task handles the final projection to avoid 'Reprojection output too large' errors."""
    combined_mask = load_base_masks()
    native_stack, native_proj = compute_native_uoi(combined_mask)
    
    target_proj = native_proj.atScale(scale)
    
    # Aggregate UOI (mean) and N (sum)
    agg_uoi = native_stack.select('UOI_mean').reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).setDefaultProjection(target_proj)
    
    agg_n = native_stack.select('N').reduceResolution(
        reducer=ee.Reducer.sum(),
        maxPixels=65535
    ).setDefaultProjection(target_proj)
    
    # Stack
    final_asset = ee.Image.cat([agg_uoi, agg_n])
    
    return final_asset, native_proj

print("\u2713 Methodological functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running Unit Tests for GEDI UOI generation...")
    test_scale = 50000
    passed = 0
    failed = 0
    
    try:
        # --- Test 1: Dual-layer mask ---
        print("  [1/5] Testing dual-layer pristine forest mask construction...")
        combined_mask = load_base_masks()
        assert isinstance(combined_mask, ee.Image), "Mask is not an ee.Image"
        passed += 1
        print("    ✓ Dual-layer mask constructed (30m native + 500m pristine)")
        
        # --- Test 2: Native UOI computation ---
        print("  [2/5] Testing native UOI computation from GEDI L2B...")
        native_stack, native_proj = compute_native_uoi(combined_mask)
        assert isinstance(native_stack, ee.Image), "Native stack is not an ee.Image"
        
        native_bands = native_stack.bandNames().getInfo()
        assert 'UOI_mean' in native_bands, f"Missing UOI_mean band. Got: {native_bands}"
        assert 'N' in native_bands, f"Missing N band. Got: {native_bands}"
        passed += 1
        print(f"    ✓ Native stack bands: {native_bands}")
        
        # --- Test 3: Full asset build ---
        print("  [3/5] Testing aggregated asset construction...")
        gedi_asset, proj = build_gedi_asset(test_scale)
        assert isinstance(gedi_asset, ee.Image), "Output is not an ee.Image"
        
        bands = gedi_asset.bandNames().getInfo()
        assert len(bands) == 2, f"Expected 2 bands, got {len(bands)}"
        assert 'UOI_mean' in bands, "Missing UOI_mean band"
        assert 'N' in bands, "Missing N band"
        passed += 1
        print(f"    ✓ Aggregated asset bands: {bands}")
        
        # --- Test 4: Deep execution ---
        print("  [4/5] Deep execution test (forces GEE to evaluate over Congo test region)...")
        tiny_test_region = ee.Geometry.Rectangle([15.0, 0.0, 15.5, 0.5])
        test_val = gedi_asset.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=tiny_test_region,
            scale=test_scale,
            maxPixels=1e9
        ).getInfo()
        
        assert isinstance(test_val, dict), "Reduction did not return a dictionary"
        assert 'UOI_mean' in test_val, "Missing UOI_mean in result"
        
        uoi_value = test_val.get('UOI_mean')
        n_value = test_val.get('N')
        
        # Sanity check: UOI must be in [0, 1]
        if uoi_value is not None:
            assert 0 <= uoi_value <= 1, f"UOI value {uoi_value} outside [0, 1] range"
        passed += 1
        print(f"    ✓ UOI mean: {uoi_value}, N (footprint count): {n_value}")
        
        # --- Test 5: N sanity check ---
        print("  [5/5] Checking footprint count is reasonable...")
        if n_value is not None:
            assert n_value > 0, f"N should be > 0 in Congo forest, got {n_value}"
            print(f"    ✓ N > 0 confirmed ({n_value} footprints in test region)")
        else:
            print(f"    ⚠ N is None (possible sparse coverage in tiny test region)")
        passed += 1
        
        print(f"\n{'='*50}")
        print(f"  ✓ ALL {passed} TESTS PASSED")
        print(f"{'='*50}")
        
    except AssertionError as e:
        failed += 1
        print(f"\n  ✗ Test Failed ({passed} passed, {failed} failed): {e}")
    except Exception as e:
        failed += 1
        print(f"\n  ✗ Unexpected Error ({passed} passed, {failed} failed): {e}")

# Execute tests
run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: VISUAL INTEGRATION TEST
# =============================================================================

def visualize_demo():
    """Generates an interactive map of GEDI UOI at 50km for quick visual QC.
    
    What to look for:
    - Green pixels (high UOI): open understory, consistent with intact megafauna
      physically maintaining forest structure. Expected in Congo.
    - Brown pixels (low UOI): dense understory, consistent with megafauna loss
      and vegetation infilling. Expected in Amazon.
    - Patchy/sparse coverage: normal for GEDI (orbital track sampling). The N band
      shows how many footprints contributed to each cell.
    """
    test_scale = 50000
    print(f"Building GEDI UOI at {test_scale/1000:.0f}km for visual inspection...")
    gedi_asset, proj = build_gedi_asset(test_scale)
    
    Map = geemap.Map(center=[0, 20], zoom=3)
    
    # UOI: sequential brown-to-green palette
    Map.addLayer(gedi_asset, {
        'bands': ['UOI_mean'],
        'min': 0.3,
        'max': 0.9,
        'palette': ['#8c510a', '#bf812d', '#dfc27d', '#f6e8c3',
                     '#c7eae5', '#80cdc1', '#35978f', '#01665e']
    }, 'GEDI UOI Mean (50km)')
    
    # Footprint density layer for QC
    Map.addLayer(gedi_asset, {
        'bands': ['N'],
        'min': 0,
        'max': 500,
        'palette': ['black', 'purple', 'blue', 'cyan', 'yellow']
    }, 'GEDI Footprint Count N', False)
    
    # Study regions
    Map.addLayer(STUDY_REGION, {'color': '0000FF'}, 'Study Regions', True)
    
    # Print summary stats per basin for quick comparison
    for basin_name, bbox in [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]:
        stats = gedi_asset.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
            geometry=bbox,
            scale=test_scale,
            maxPixels=1e9
        ).getInfo()
        print(f"  {basin_name}: UOI_mean={stats.get('UOI_mean_mean', 'N/A'):.4f}, "
              f"UOI_std={stats.get('UOI_mean_stdDev', 'N/A'):.4f}, "
              f"N_mean={stats.get('N_mean', 'N/A'):.1f}")
    
    return Map

# Display the map
visualize_demo()

In [ ]:
# =============================================================================
# BLOCK 5: EXECUTION (ASSET EXPORT)
# =============================================================================

def export_all_scales(dry_run=True):
    tasks = []
    print(f"Configuring export tasks for {len(SCALES)} scales...")
    
    for scale in SCALES:
        gedi_asset, proj = build_gedi_asset(scale)
        
        task = ee.batch.Export.image.toAsset(
            image=gedi_asset,
            description=f'GEDI_{scale}_Export',
            assetId=f'{ASSET_ROOT}/GEDI_{scale}',
            region=STUDY_REGION.geometry(),
            scale=scale,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append(task)
        
        if not dry_run:
            task.start()
            
    print(f"\u2713 Configured {len(tasks)} export tasks.")
    if dry_run:
        print("DRY RUN: Tasks created but not started. Call export_all_scales(dry_run=False) to begin processing on GEE servers.")
    else:
        print("Tasks started! Monitor progress in the GEE Code Editor or via ee.batch.Task.list()")

# To execute the exports, set dry_run=False
export_all_scales(dry_run=True)